# Differential regulation: ANOVA across more than two groups and posthoc t-tests

This API example shows the functionality in the
[`acore.differential_regulation`](acore.differential_regulation) module:

- Analysis of Variance (ANOVA) as omnibus test across more than two groups
- posthoc t-tests between each of the groups

An omnibus analysis across groups
is combined with posthoc analysis between each set of the separate groups.
The omnibus test's adjusted p-value and rejection flag are in `padj` and
`rejected`; pairwise posthoc results use `posthoc pvalue adj` and
`posthoc rejected`.
`Method` records the selected omnibus test (`One-way anova` here); use
`run_ancova` when covariates should be included in the omnibus model.

The function is the same as for the two groups case. The `group1` and
`group2` columns give the posthoc comparison.


In [ ]:
%pip install acore

In [ ]:
import dsp_pandas
import pandas as pd
import vuecore.plots.basic.scatter

import acore.differential_regulation as ad

dsp_pandas.format.set_pandas_options(
    max_columns=9,
    max_colwidth=20,
)

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/Multiomics-Analytics-Group/acore/"
    "main/example_data/MTBLS13311/"
)
fname: str = "MTBLS13411_omics_and_meta.csv"  # combined omics and meta data
covariates: list[str] = []
group: str = "group"
drop_cols: list[str] = ["Factor Value[Strain type]"]
subject_col: str | int = 0
factor_and_covars: list[str] = [group, *covariates]

Use combined dataset for ANOVA analysis.

In [ ]:
omics_and_meta = (
    pd.read_csv(f"{BASE}/{fname}", index_col=subject_col)
    .convert_dtypes()
    .dropna(subset=factor_and_covars)
)
omics_and_meta

Drop unnecessary columns, if there are any specified in `drop_cols`.

In [ ]:
if drop_cols:
    omics_and_meta.drop(columns=drop_cols, inplace=True)
omics_and_meta

Check data types of the columns. Metadata can be numeric, but also strings.

In [ ]:
omics_and_meta.dtypes.value_counts()

In [ ]:
omics_and_meta[factor_and_covars]

# Run ANOVA with four groups
Acore make each combinatorial comparison between groups in the group column.

In [ ]:
if isinstance(subject_col, int):
    subject_col = omics_and_meta.index.name
anova = (
    ad.run_anova(
        omics_and_meta,  # .reset_index(),
        subject=None,
        drop_cols=[],
        group=group,
    ).set_index(["identifier", "group1", "group2"])
    # .sort_values(by="padj")
)
anova.head().T

## pairwise t-test results:

In [ ]:
cols_pairwise_ttest = [
    # "group1",
    # "group2",
    "mean(group1)",
    "std(group1)",
    "mean(group2)",
    "std(group2)",
    "posthoc Paired",
    "posthoc Parametric",
    "posthoc T-Statistics",
    "posthoc dof",
    "posthoc tail",
    "posthoc pvalue",
    "posthoc pvalue adj",
    "posthoc rejected",
    "posthoc BF10",
    "posthoc effsize",
    # "identifier",
    "log2FC",
    "FC",
    "efftype",
]
anova[cols_pairwise_ttest].T

ANOVA results

In [ ]:
anova.drop(columns=cols_pairwise_ttest)

Test results

In [ ]:
regex_filter = "pval|padj|reject|stat|FC"
view = anova.filter(regex=regex_filter)
view

## Posthoc volcano plot: WRP versus WT
Select one pairwise comparison so each identifier has one point.
The x-axis shows its `log2FC`, the y-axis its posthoc p-value, and
the color its posthoc FDR significance decision (`posthoc rejected`).
Change `comparison` to another pair from the `group1` and `group2` index levels.

In [ ]:
comparison = ("WRP", "WT")
pairwise_results = anova.xs(
    comparison, level=["group1", "group2"], drop_level=False
).reset_index()

scatter_plot_adv = vuecore.plots.basic.scatter.create_scatter_plot(
    data=pairwise_results,
    x="log2FC",
    y="-log10 posthoc pvalue",
    color="posthoc rejected",
    title="Posthoc Volcano Plot",
    subtitle=f"{comparison[0]} versus {comparison[1]}",
    labels={
        "log2FC": f"log2 fold change ({comparison[0]} / {comparison[1]})",
        "-log10 posthoc pvalue": "-log10(posthoc p-value)",
        "posthoc rejected": "Posthoc FDR corrected significant",
        "identifier": "Protein Identifier",
    },
    hover_data=["identifier", "group1", "group2"],
    # currently does not work:
    # color_discrete_map={False: "#2166AC", True: "#B2182B"},  # Blue  # Red
    color_discrete_sequence=["red", "blue"],
    opacity=1,
    marker_line_width=1,
    marker_line_color="darkgray",
    width=800,
    height=600,
)
scatter_plot_adv